<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/main/notebooks/day_two_de.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Tag Zwei in Colab öffnen"/></a>

# Tag Zwei: Stellt eure eigene Forschungsfrage

Heute führt eure Gruppe eine kleine fMRT-Analyse zu einer Frage durch, die ihr selbst auswählt.

Ihr werdet:

- zwei Aufgabenbedingungen auswählen und vergleichen,
- eine Vorhersage aufschreiben,
- einen Kontrast visualisieren, und
- erklären, was das Ergebnis bedeuten könnte.

Der Code ist weiterhin angeleitet. Eure Aufgabe ist es, die wissenschaftlichen Entscheidungen zu treffen und das Ergebnis vorsichtig zu interpretieren.

Gestern haben wir ein Beispiel untersucht: Tastendrücke mit der rechten im Vergleich zur linken Hand.

Heute wählt jede Gruppe eine eigene Frage. Zum Beispiel:

- **Sehen:** Welche Areale reagieren stärker auf visuelle Muster?
- **Hören:** Welche Areale reagieren stärker, wenn Anweisungen für eine Aufgabe gehört statt gesehen werden?
- **Sprache:** Welche Areale reagieren stärker auf Sätze als auf einfache visuelle Muster?
- **Rechnen (Challenge):** Was verändert sich im Gehirn, wenn Menschen rechnen, statt Sätze zu verstehen?

Bevor ihr die Analyse ausführt, sollte sich eure Gruppe auf eine Frage und eine Vorhersage einigen.

## Setup

Führe diese Zelle zuerst aus. Sie installiert die nötigen Werkzeuge, lädt die fMRT-Daten und lädt die anatomische Vorlage.

In [ ]:
# @title Setup: Werkzeuge installieren, Pakete importieren und Daten laden
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy", "pandas"
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting
from nilearn.glm.first_level import FirstLevelModel

%matplotlib inline

anat_img = datasets.load_mni152_template(resolution=2)
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)
events = pd.read_table(fmri_dataset.events)

print("Setup abgeschlossen.")
print("fMRT-Bildform:", fmri_img.shape)
print("Anzahl der Ereignisse:", len(events))

## Wählt aus diesen Aufgabenbedingungen

Das Localizer-Experiment verwendet diese Bedingungen. Ihr könnt sie genau so in die nächste Codezelle kopieren:

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/pinel_localizer_tasks.png" alt="Beispiele für visuelle und auditive Aufgaben im Pinel-Localizer-Experiment" width="850">


```text
horizontal_checkerboard
vertical_checkerboard
sentence_reading
sentence_listening
visual_computation
audio_computation
```

Die meisten Gruppen sollten mit Hören, Vision oder Sprache beginnen. Die Rechenbedingungen sind Challenge-Optionen, weil sie schwieriger zu interpretieren sein können.

## Wählt eure Forschungsfrage

Arbeitet in eurer Gruppe. Wählt eine Frage, bevor ihr den Code verändert.

Empfohlene Einstiegsfragen:

1. **Hören:** `sentence_listening` verglichen mit `sentence_reading`.
2. **Sehen:** `horizontal_checkerboard` verglichen mit `vertical_checkerboard`.
3. **Sprache:** `sentence_reading` verglichen mit `horizontal_checkerboard` oder `sentence_listening` verglichen mit `horizontal_checkerboard`.
4. **Challenge:** `visual_computation` verglichen mit `sentence_reading` oder `audio_computation` verglichen mit `sentence_listening`.


Manche Vergleiche sind leichter zu interpretieren als andere. Das ist in der fMRT-Forschung normal. Ein Kontrast kann einen klaren farbigen Cluster zeigen, aber der Grund für diesen Cluster kann trotzdem kompliziert sein.

Bevor ihr weitermacht, besprecht diese vier Punkte als Gruppe:

- Welche zwei Bedingungen vergleicht ihr?
- Für welche Bedingung erwartet ihr stärkere Aktivität?
- Wo im Gehirn erwartet ihr die stärkste Aktivität: vorne/hinten, links/rechts, oben/unten?
- Wie kommt ihr zu eurer Erwartung?

## Wählt Condition A und Condition B

Ein Kontrast vergleicht zwei Bedingungen:

```text
Condition A - Condition B
```

Warme Farben bedeuten: mehr Aktivität für **Condition A** als für **Condition B**.

Kühle Farben bedeuten: mehr Aktivität für **Condition B** als für **Condition A**.

In der nächsten Zelle könnt ihr das Beispiel behalten oder die Bedingungen in den Anführungszeichen ersetzen.

In [ ]:
# Wählt zwei Bedingungen.
# Kopiert die Bedingungsnamen genau und lasst die Anführungszeichen stehen.

condition_a = "sentence_listening"
condition_b = "sentence_reading"

my_contrast = condition_a + " - " + condition_b

print("Condition A:", condition_a)
print("Condition B:", condition_b)
print("Mein Kontrast:", my_contrast)

<details>
<summary>Braucht ihr Ideen? Klickt hier für Beispiele </summary>

```python
# Hören und Sprache: gehörte Sätze verglichen mit gelesenen Sätzen
condition_a = "sentence_listening"
condition_b = "sentence_reading"

# Vision: ein visuelles Muster verglichen mit einem anderen visuellen Muster
condition_a = "horizontal_checkerboard"
condition_b = "vertical_checkerboard"

# Sprache: Sätze lesen verglichen mit einem einfachen visuellen Muster
condition_a = "sentence_reading"
condition_b = "horizontal_checkerboard"

# Sprache und Hören: Sätze hören verglichen mit einem visuellen Muster
condition_a = "sentence_listening"
condition_b = "horizontal_checkerboard"

# Challenge: visuelle Rechenaufgaben verglichen mit Sätze lesen
# Das kann Zahlen betreffen, aber auch Aufmerksamkeit und Aufgabenschwierigkeit.
condition_a = "visual_computation"
condition_b = "sentence_reading"

# Challenge: gehörte Rechenaufgaben verglichen mit Sätze hören
# Das kann Zahlen betreffen, aber auch Aufmerksamkeit und Aufgabenschwierigkeit.
condition_a = "audio_computation"
condition_b = "sentence_listening"

# Schwieriger: Sätze hören verglichen mit einem anderen visuellen Muster
# Das mischt Hören/Sehen, Sprache und Unterschiede im visuellen Muster.
condition_a = "sentence_listening"
condition_b = "vertical_checkerboard"
```

Wählt ein Paar aus. Führt nicht alle Beispiele gleichzeitig aus.

</details>

## Stop and Think

Geht nicht zu schnell über diesen Teil hinweg. Diskutiert in eurer Gruppe:

1. Was bedeuten warme Farben bei eurem Kontrast?
2. Was bedeuten kühle Farben?
3. Wo erwartet ihr die stärksten warmen Farben: vorne/hinten, links/rechts, oben/unten?
4. Was könnten die Farben bedeuten: gibt es nur eine Schlussfolgerung oder unterscheiden sich die beiden Bedingungen noch auf andere Art?

Schreibt eine Vorhersage auf, bevor ihr weitermacht.

## Das Modell aufstellen

Jetzt prüft Nilearn jedes Voxel.

Das kann einen kurzen Moment dauern. Diese Zelle müsst ihr nicht verändern.

In [ ]:
first_level_model = FirstLevelModel(
    t_r=2.4,
    slice_time_ref=0,
    hrf_model="spm",
    drift_model="cosine",
    high_pass=0.01,
    smoothing_fwhm=4,
    minimize_memory=True,
)

first_level_model = first_level_model.fit(fmri_img, events=events)
print("Modell erfolgreich angepasst.")

## Euren Kontrast berechnen

Jetzt berechnen wir den Kontrast, den ihr oben ausgewählt habt.

In [ ]:
my_map = first_level_model.compute_contrast(
    my_contrast,
    output_type="z_score",
)

print("Kontrast berechnet:", my_contrast)

## Den Kontrast visualisieren

Das Bild unten visualisiert den Kontrast. Ihr könnt wieder interaktiv darin herumklicken.

Denkt daran:

- Warme Farben bedeuten **Condition A > Condition B**.
- Kühle Farben bedeuten **Condition B > Condition A**.

Versucht die Orte im Gehirn mit der stärksten Aktivität zu finden!

In [ ]:
result_viewer = plotting.view_img(
    my_map,
    bg_img=anat_img,
    threshold=3.0,
    title=my_contrast,
)

result_viewer

## Prüft euren Punkt, die stärkste Stelle und eure erwartete Region

Benutzt zuerst die interaktive Ansicht oben. Klickt auf eine stark farbige Stelle und schreibt euch die `x`-, `y`- und `z`-Koordinaten auf, die in der Ansicht angezeigt werden.

Tragt diese Koordinaten in die kleine Codezelle unten ein. Sie müssen nicht perfekt sein.

Wählt dann die Gehirnregion, die zu eurer Vorhersage passt:

Benutzt `"Occipital"`, `"Temporal"`, `"Parietal"` oder `"Frontal"`.

In [ ]:
# Tragt die Koordinaten aus der interaktiven Ansicht ein.
# Ändert die Zahlen passend zu euren x-, y- und z-Werten.
student_x = 0  # @param {type:"number"}
student_y = 0  # @param {type:"number"}
student_z = 0  # @param {type:"number"}

student_xyz = [student_x, student_y, student_z]
print("Eure Koordinate aus der interaktiven Ansicht:", student_xyz)

In [ ]:
# Wählt die Region, die zu eurer Vorhersage passt.
# Optionen: "Occipital", "Temporal", "Parietal", "Frontal"
expected_region = "Occipital"  # @param ["Occipital", "Temporal", "Parietal", "Frontal"]

print("Erwartete Region:", expected_region)

In [ ]:
# @title Atlas-Abfrage: euren Punkt prüfen, dann automatische Peaks finden und plotten
from nilearn.reporting import get_clusters_table
from nibabel.affines import apply_affine

atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
atlas_img = image.load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
atlas_labels = atlas.labels


def atlas_label_for_xyz(xyz):
    voxel = np.round(
        apply_affine(np.linalg.inv(atlas_img.affine), xyz)
    ).astype(int)

    if np.any(voxel < 0) or np.any(voxel >= atlas_data.shape):
        return "außerhalb des Atlas"

    label_number = int(atlas_data[tuple(voxel)])
    if label_number == 0:
        return "kein klares Atlas-Label"

    return atlas_labels[label_number]


def atlas_label_for_peak(row):
    xyz = [row["X"], row["Y"], row["Z"]]
    return atlas_label_for_xyz(xyz)


print("Eure Koordinate aus der interaktiven Ansicht:")
print("  Koordinate:", student_xyz)
print("  Atlas-Label:", atlas_label_for_xyz(student_xyz))

plotting.plot_stat_map(
    my_map,
    bg_img=anat_img,
    threshold=3.0,
    cut_coords=student_xyz,
    title="Eure Koordinate aus der interaktiven Ansicht",
    display_mode="ortho",
)

cluster_table = get_clusters_table(
    my_map,
    stat_threshold=3.0,
    cluster_threshold=10,
)

if len(cluster_table) == 0:
    print()
    print("Bei diesem Schwellenwert wurde kein starker Cluster gefunden.")
    print("Schaut in die interaktive Karte und beschreibt vorsichtig, was ihr seht.")
else:
    atlas_table = cluster_table.copy()
    atlas_table["atlas_label"] = atlas_table.apply(atlas_label_for_peak, axis=1)

    strongest_peak = atlas_table.iloc[0]
    strongest_xyz = [strongest_peak["X"], strongest_peak["Y"], strongest_peak["Z"]]

    print()
    print("Stärkster Peak insgesamt:")
    print("  Koordinate:", strongest_xyz)
    print("  Atlas-Label:", strongest_peak["atlas_label"])

    plotting.plot_stat_map(
        my_map,
        bg_img=anat_img,
        threshold=3.0,
        cut_coords=strongest_xyz,
        title="Stärkster Peak insgesamt: " + strongest_peak["atlas_label"],
        display_mode="ortho",
    )

    expected_peaks = atlas_table[
        atlas_table["atlas_label"].str.contains(expected_region, case=False, na=False)
    ]

    if len(expected_peaks) == 0:
        print()
        print("Kein klarer Peak in der erwarteten Region gefunden:", expected_region)
        print("Es werden nur eure Koordinate und der stärkste Peak insgesamt geplottet.")
    else:
        expected_peak = expected_peaks.iloc[0]
        expected_xyz = [expected_peak["X"], expected_peak["Y"], expected_peak["Z"]]

        print()
        print("Stärkster Peak in der erwarteten Region:", expected_region)
        print("  Koordinate:", expected_xyz)
        print("  Atlas-Label:", expected_peak["atlas_label"])

        plotting.plot_stat_map(
            my_map,
            bg_img=anat_img,
            threshold=3.0,
            cut_coords=expected_xyz,
            title="Stärkster Peak in der erwarteten Region: " + expected_peak["atlas_label"],
            display_mode="ortho",
        )

## Euer Ergebnis interpretieren

Diskutiert euer Ergebnis als Gruppe. Nutzt die Fragen unten, um eure Interpretation zu ordnen.

### Euer Kontrast

- Welche Bedingung war Condition A?
- Welche Bedingung war Condition B?
- Was bedeuteten warme Farben bei eurem Kontrast?
- Was bedeuteten kühle Farben bei eurem Kontrast?

### Wo ihr Aktivität gefunden habt

Benutzt die interaktive Ansicht, um die stärksten Cluster zu beschreiben. Ihr müsst nicht perfekt sein. Macht eine vorsichtige, gut begründete Vermutung.

Kurzer Guide:

- **Okzipitallappen:** hinten im Gehirn, oft wichtig für Sehen.
- **Temporallappen:** seitlich/unterer Teil des Gehirns, oft wichtig für Hören und Sprache.
- **Parietallappen:** oben/hinten im Gehirn, oft wichtig für Aufmerksamkeit, Raum und zahlenbezogene Aufgaben.
- **Frontallappen:** vorne im Gehirn, oft wichtig für Planung, Aufmerksamkeit und schwierige Aufgaben.

Für die stärksten warmen Farben besprecht:

- Wo liegen die Areale?
- Was ist eure beste Vermutung bzgl. der Gehirnlappen?
- Welche Funktion könnte zu diesem Lappen passen?

Für die stärksten kühlen Farben besprecht dieselben Fragen.

## Vorsichtig sein

Das sind echte fMRT-Daten, aber unsere Analyse ist vereinfacht.

- Wir haben nur eine Versuchsperson verwendet.
- Nicht jeder farbige Punkt ist automatisch eine Entdeckung.
- Manche Kontraste sind leichter zu interpretieren als andere.
- Ein Kontrast kann mehr als eine Sache gleichzeitig vergleichen.

Zum Beispiel ist `horizontal_checkerboard - vertical_checkerboard` recht klar: Die beiden Aufgaben sind sehr ähnlich, aber das visuelle Muster verändert sich. `sentence_listening - sentence_reading` verändert dagegen mehr als eine Sache: Hören gegen Sehen, gesprochene Sprache gegen geschriebene Sprache und vielleicht Aufmerksamkeit. Die farbigen Cluster können trotzdem interessant sein, aber wir müssen vorsichtig sein, was wir behaupten.

Die Rechenkontraste sind besonders schwierig. Wenn ihr `visual_computation - sentence_reading` vergleicht, könnte ein Cluster mit Zahlen zusammenhängen. Er könnte aber auch damit zusammenhängen, dass Rechnen schwieriger ist, mehr Aufmerksamkeit braucht oder Gedächtnis beansprucht. Deshalb sollten wir sagen: "Dieses Areal könnte an Berechnung beteiligt sein" und nicht: "Das ist das Mathe-Areal".

Das ist ein häufiges Problem in der fMRT-Forschung. Ein Kontrast erklärt sich nicht selbst. Forschende müssen fragen: **Was genau haben wir verglichen? Was hat sich zwischen den beiden Bedingungen noch verändert? Könnte es eine andere Erklärung geben?**

Gute Wissenschaft bedeutet, sagen zu können, was ein Ergebnis nahelegt und worüber man noch unsicher ist.

## Bereitet euer Poster vor

Jede Gruppe sollte bereit sein zu erklären:

1. Unsere Forschungsfrage war...
2. Wir verglichen ... minus ...
3. Wir sagten voraus...
4. Warme Farben bedeuteten...
5. Kühle Farben bedeuteten...
6. Wir fanden...
7. Der Atlas deutete auf...
8. Wir sind noch unsicher bei...

## Quellen

- Pinel, P., Thirion, B., Meriaux, S. et al. (2007). Fast reproducible identification and large-scale databasing of individual functional cognitive networks. *BMC Neuroscience, 8*, 91. https://doi.org/10.1186/1471-2202-8-91
- [Nilearn First-Level-Modelle](https://nilearn.github.io/stable/glm/first_level_model.html)
- [Nilearn-Plotting-Werkzeuge](https://nilearn.github.io/stable/modules/plotting.html)
- [Nilearn-Atlanten](https://nilearn.github.io/stable/modules/datasets.html#atlases)